In [1]:
import json
import os
import glob
import re
import warnings
from getpass import getpass
from langchain_core.documents import Document
warnings.filterwarnings("ignore")

print("="*50)
print(" STEP 1: SMART INGESTION & VECTOR DB")
print("="*50)


os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
for k in ['HTTP_PROXY', 'HTTPS_PROXY', 'http_proxy', 'https_proxy', 'ALL_PROXY', 'all_proxy']:
    os.environ.pop(k, None)

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else "/home/hardy/ccs_project"
RAG_DOCS_DIR = os.path.join(PROJECT_ROOT, "data/rag_documents")
CHROMA_DB_DIR = os.path.join(PROJECT_ROOT, "vectorstore")

if not os.path.exists(RAG_DOCS_DIR):
    raise FileNotFoundError(f" RAG document directory not found: {RAG_DOCS_DIR}")


def clean_text(text: str) -> str:
 
    if not text: return ""
    text = re.sub(r'\s+', ' ', text) 
    text = re.sub(r'Page \d+ of \d+', ' ', text, flags=re.IGNORECASE) 
    text = re.sub(r'\[\d+\]', ' ', text) 
    text = re.sub(r'[-_=]{3,}', ' ', text) 
    return text.strip()

def clean_documents(docs):

    cleaned_docs = []
    for d in docs:
        cleaned = clean_text(d.page_content)
        if len(cleaned) >= 80:  
            cleaned_docs.append(Document(page_content=cleaned, metadata=d.metadata))
    return cleaned_docs

def classify_doc(filename: str) -> str:
    name = filename.lower()

    if any(k in name for k in ["bachu", "screening", "criteria", "srccs"]):
        return "ccs_rules"

    elif any(k in name for k in ["seal", "caprock", "evaporite", "gypsum"]):
        return "seal"

    elif any(k in name for k in ["tarim", "tectonic", "structural", "basin"]):
        return "tarim_geology"

    else:
        return "ccs_review"


print(" Initializing Embedding Model (BAAI/bge-small-en-v1.5)...")
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

if os.path.exists(CHROMA_DB_DIR) and os.listdir(CHROMA_DB_DIR):
    print(" Loading existing Chroma database...")
    print("   (Note: Delete 'vectorstore' folder and rerun to rebuild if you added new PDFs)")
    vectorstore = Chroma(persist_directory=CHROMA_DB_DIR, embedding_function=embedding_model)
else:
    print(" Building new Chroma database...")
    pdf_files = glob.glob(os.path.join(RAG_DOCS_DIR, "*.pdf"))
    if not pdf_files: raise ValueError(f" No PDFs found in {RAG_DOCS_DIR}!")

    all_docs = []
    for pdf_file in pdf_files:
        filename = os.path.basename(pdf_file)
        print(f"   - Loading & Cleaning: {filename}")
        try:
            loader = PyMuPDFLoader(pdf_file)
            docs = loader.load()
            docs = clean_documents(docs)


            doc_type = classify_doc(filename)
            for d in docs:
                d.metadata["source"] = pdf_file
                d.metadata["filename"] = filename
                d.metadata["doc_type"] = doc_type

            if docs:
                all_docs.extend(docs)
                print(f"     -> Classified as: [{doc_type}], Extracted {len(docs)} valid pages.")
            else:
                print(f"      WARNING: '{filename}' has very little usable text.")
        except Exception as e:
            print(f"      ERROR loading {filename}: {e}")

    if not all_docs: raise ValueError(" FATAL: No usable text extracted from PDFs!")


    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=700, 
        chunk_overlap=120,
        separators=["\n\n", "\n", ". ", "; ", ", ", " "]
    )
    chunks = text_splitter.split_documents(all_docs)
    print(f" Split into {len(chunks)} high-quality chunks.")

    print("\n Chunk Quality Check (First 300 chars):")
    print("-" * 40)
    print(chunks[0].page_content[:300] + "...")
    print("-" * 40)

    vectorstore = Chroma.from_documents(documents=chunks, embedding=embedding_model, persist_directory=CHROMA_DB_DIR)

print("\n Phase 6 - Step 1 Complete! Vector Database is primed for multi-path retrieval.")


 STEP 1: SMART INGESTION & VECTOR DB
 Initializing Embedding Model (BAAI/bge-small-en-v1.5)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Building new Chroma database...
   - Loading & Cleaning: Tarim_Carbonate_Caprock_Distribution_Study.pdf
     -> Classified as: [seal], Extracted 14 valid pages.
   - Loading & Cleaning: Raza_2016_CO2_Storage_Site_Screening_Criteria.pdf
     -> Classified as: [ccs_rules], Extracted 9 valid pages.
   - Loading & Cleaning: Comprehensive_Review_CO2_Geological_Storage_2024.pdf
     -> Classified as: [ccs_review], Extracted 13 valid pages.
   - Loading & Cleaning: Tarim_Gypsum_Salt_Caprock_Sealing_Capacity_2022.pdf
     -> Classified as: [seal], Extracted 19 valid pages.
   - Loading & Cleaning: Benson_Cole_2008_CO2_Sequestration_Deep_Sedimentary_Formations.pdf
     -> Classified as: [ccs_review], Extracted 6 valid pages.
   - Loading & Cleaning: CCS_Lessons_and_Prospects_Review_2014.pdf
     -> Classified as: [ccs_review], Extracted 20 valid pages.
   - Loading & Cleaning: Bachu_2003_Screening_Ranking_Sedimentary_Basins.pdf
     -> Classified as: [ccs_rules], Extracted 13 valid pages.
   -

In [2]:
import os
import json
import time
import pandas as pd
import numpy as np
from getpass import getpass
from langchain_openai import ChatOpenAI 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

print("="*50)   
print("  STEP 3: DIVERSE GRADIENT RAG SCREENING & THESIS EXPORT")
print("="*50)

for env_var in ['HTTP_PROXY', 'HTTPS_PROXY', 'http_proxy', 'https_proxy', 'ALL_PROXY', 'all_proxy']:
    os.environ.pop(env_var, None)

if "DEEPSEEK_API_KEY" not in os.environ:
    os.environ["DEEPSEEK_API_KEY"] = getpass(" Enter your DeepSeek API Key: ")

print(" Initializing DeepSeek LLM...")
llm = ChatOpenAI(
    model='deepseek-chat', 
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url='https://api.deepseek.com',
    temperature=0.1,
    max_tokens=1000, 
    timeout=60
)

SOURCE_NAME_MAP = {
    "IPCC_SRCCS_Underground_Geological_Storage.pdf": "IPCC SRCCS",
    "IEAGHG_Seal_Integrity_Review_Sections.pdf": "IEAGHG Seal Integrity Review",
    "Bachu_2003_Screening_Ranking_Sedimentary_Basins.pdf": "Bachu 2003",
    "Raza_2016_CO2_Storage_Site_Screening_Criteria.pdf": "Raza et al. 2016",
    "Tarim_Tectonic_Framework_Evolution.pdf": "Tarim Tectonic Framework",
    "Central_Tarim_Basin_Thermal_History_Case_Study.pdf": "Central Tarim Thermal History",
    "CO2_Geological_Storage_Review_2024.pdf": "CO2 Geological Storage Review 2024",
    "CCS_Review_Lessons_and_Prospects_2014.pdf": "CCS Review 2014",
    "Benson_Cole_2008_CO2_Sequestration_Deep_Sedimentary_Formations.pdf": "Benson & Cole 2008",
    "Tarim_Gypsum_Salt_Caprock_Sealing_Capacity_2022.pdf": "Tarim Gypsum-Salt Caprock Study",
    "Tarim_Carbonate_Caprock_Distribution_Study.pdf": "Tarim Carbonate Caprock Study"
}

def format_docs_with_citations(docs):
    formatted = []
    for doc in docs:
        filename = doc.metadata.get("filename", os.path.basename(doc.metadata.get("source", "Unknown")))
        source = SOURCE_NAME_MAP.get(filename, filename) 
        page = doc.metadata.get("page", 0) + 1
    
        content = doc.page_content[:800].replace('\n', ' ') 
        formatted.append(f"[Source: {source}, Page: {page}]\n{content}...")
    return "\n\n---\n\n".join(formatted)

def estimate_evidence_confidence(unique_docs):
    n = len(unique_docs)
    if n >= 8: return "High"
    elif n >= 5: return "Medium"
    elif n >= 2: return "Low"
    return "Very Low"

strict_prompt_template = """
You are a Principal Lead Geologist at an international energy corporation, specializing in Carbon Capture and Storage (CCS) site pre-screening. 
Your task is to write a highly professional, bespoke, and extremely detailed technical evaluation report for a candidate geological interval in the {basin}.

[CRITICAL ENGINEERING PHILOSOPHY]:
This is a Pre-Screening system. Your goal is to identify ALL viable candidates to prevent missing high-potential reservoirs (False Negatives), while explicitly flagging containment risks (False Positives) for downstream human evaluation.

[CRITICAL INSTRUCTION ON THICKNESS]: 
The "Thickness" provided is a proxy measured in "log samples" ({unit}), NOT meters. Assume standard logging resolution (e.g., 0.1524m per sample). 
Therefore, 65 samples ≈ 10 meters. 
Use 65 samples as your strict baseline for "adequate volumetric capacity". If a target is below 30 samples, explicitly flag it as a "severe volumetric constraint".

==================================================
INPUT DATA PROFILE
Target ID: {interval_id}
Depth Range: {top} - {bottom}
Machine Learning Prediction Confidence: {ml_conf} (Range: 0-1, >0.8 is strong, <0.7 is weak)
Thickness Proxy: {thickness} {unit}
Petrophysical Log Averages: {petrophysics}
Retrieved Geological Context (RAG): 
{context}
==================================================

TASK REQUIREMENTS:
1. Synthesize the provided data meticulously. Cross-validate the ML Confidence with the specific Petrophysical logs and Retrieved Context.
2. Ground your analysis strictly in the provided "Retrieved Geological Context". You MUST cite the specific [Source: XX, Page: YY] when referencing seal integrity or fault risks.
3. Assess the tradeoff: Do not outright reject a highly confident ML prediction due to minor structural risks, but explicitly log the risk in your recommendation.
4. ABSOLUTE RULE: Even if the interval is extremely thin and has a fatal volumetric flaw, you MUST STILL evaluate the regional Caprock and Structural Risk based on the retrieved context. DO NOT OUTPUT "N/A" for any field.

YOUR OUTPUT MUST EXACTLY FOLLOW THIS FORMAT (Do not add extra formatting or markdown outside these fields):

Evidence Type: [Choose one: Robust / Regional / Insufficient]
Caprock Evidence: [Choose one: Strong / Moderate / Weak. NEVER use N/A.]
Structural Risk Evidence: [Choose one: Low Risk / Moderate Risk / High Risk. NEVER use N/A.]
Screening Verdict: [Choose one: High / Medium-High / Medium / Medium-Low / Low]
Reason: [Write a comprehensive, highly analytical, in-depth evaluation (approx. 150-250 words) formatted strictly as ONE SINGLE CONTINUOUS PARAGRAPH. Do NOT use line breaks or bullet points here. You MUST logically synthesize three layers: 1) The synergy between ML Confidence and specific Petrophysical log values (e.g., GR, NPHI), 2) Volumetric Capacity based on Thickness constraints, and 3) Containment Risk incorporating Caprock and Fault evaluations. MUST include [Source: XX, Page: YY] citations.]
Recommendation: [Write a detailed, multi-step actionable recommendation (approx. 3-5 sentences) formatted strictly as ONE SINGLE CONTINUOUS PARAGRAPH. Specify both immediate engineering next steps (e.g., high-resolution 3D seismic acquisition, core sampling) and the overarching strategic justification for archiving or advancing the target.]
"""

prompt = PromptTemplate.from_template(strict_prompt_template)
llm_chain = prompt | llm | StrOutputParser()
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else "/home/hardy/ccs_project"
REPORT_DIR = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(REPORT_DIR, exist_ok=True)
JSON_INPUT_PATH = os.path.join(REPORT_DIR, "rag_target_intervals.json")

with open(JSON_INPUT_PATH, 'r') as f: ml_data = json.load(f)

all_targets = ml_data.get("identified_ccs_targets", [])
if not all_targets: raise ValueError(" No identified_ccs_targets found.")

sorted_targets = sorted(all_targets, key=lambda x: x.get("average_confidence", 0), reverse=True)
targets = sorted_targets

print(f" Loaded {len(all_targets)} intervals. Evaluating targets via RAG Pipeline...")
final_reports = []
summary_table_data = [] 

def search_by_type(vectorstore, query, doc_type=None, k=4):
    try:
        if doc_type: return vectorstore.similarity_search(query, k=k, filter={"doc_type": doc_type})
        return vectorstore.similarity_search(query, k=k)
    except Exception:
        return []

for i, target in enumerate(targets):
    interval_id = target.get("interval_id", f"Target_{i+1}")
    top, bottom = target.get("top_depth", "N/A"), target.get("bottom_depth", "N/A")
    basin = target.get("basin", "Unknown Basin")
    unit = target.get("thickness_unit", "log samples")
    ml_conf = target.get("average_confidence", 0.0)
    thickness = target.get("thickness", 0.0)
    
    petrophysics_dict = target.get("petrophysical_signatures", {})
    if petrophysics_dict:
        petro_str = ", ".join([f"{k}: {v}" for k, v in petrophysics_dict.items()])
    else:
        petro_str = "Not available"
    
    print(f"\n" + "-"*50)
    if i == 0: tier = " BEST"
    elif i < 5: tier = " Top Tier"
    elif i < 15: tier = " Middle Tier"
    else: tier = " Lower Tier"
    print(f"  DEEP EVALUATING: {interval_id} | {tier}")
    print(f"    (ML Conf: {ml_conf:.4f}, Thick: {thickness}, Petro: {petro_str})")
    
    query_rules = "CCS site screening criteria depth 800 m porosity permeability seal integrity fault risk"
    query_caprock = f"{basin} caprock seal mudstone shale evaporite gypsum porosity permeability"
    query_risk = f"{basin} fault tectonic fracture structural stability strike-slip anticline"
    
    docs_rules = search_by_type(vectorstore, query_rules, doc_type="ccs_rules", k=3)
    docs_caprock = search_by_type(vectorstore, query_caprock, doc_type="seal", k=4)
    docs_risk = search_by_type(vectorstore, query_risk, doc_type="tarim_geology", k=4)
    
    unique_docs = []
    seen = set()
    for d in docs_rules + docs_caprock + docs_risk:
        key = (d.metadata.get("filename"), d.metadata.get("page"), d.page_content[:100])
        if key not in seen:
            seen.add(key)
            unique_docs.append(d)
            
    evidence_conf = estimate_evidence_confidence(unique_docs)
    
    if not unique_docs:
        answer = "Evidence Type: Insufficient\nCaprock Evidence: Unknown\nStructural Risk Evidence: Unknown\nScreening Verdict: Uncertain\nReason: No relevant documents retrieved [Source: N/A, Page: N/A].\nRecommendation: Require manual geological review."
        retrieved_sources = []
    else:
        retrieved_sources = [{"source": SOURCE_NAME_MAP.get(d.metadata.get("filename", "Unknown"), d.metadata.get("filename", "Unknown")), "page": d.metadata.get("page", 0) + 1, "snippet": d.page_content[:150]} for d in unique_docs]
        retrieved_sources = [dict(t) for t in {tuple(d.items()) for d in retrieved_sources}]
        
        context_str = format_docs_with_citations(unique_docs)
        print(f" Evaluating via DeepSeek...")
        try:
            answer = llm_chain.invoke({
                "basin": basin,
                "interval_id": interval_id,
                "top": top, 
                "bottom": bottom, 
                "ml_conf": round(ml_conf, 4), 
                "thickness": thickness,
                "unit": unit, 
                "petrophysics": petro_str,
                "context": context_str
            })
        except Exception as e:
            answer = f"Evidence Type: Error\nCaprock Evidence: Error\nStructural Risk Evidence: Error\nScreening Verdict: Uncertain\nReason: API Error - {e}\nRecommendation: Review API status."

    print(f"\n Report:\n{answer}")
    
    parsed_fields = {
        "Evidence Type": "Unknown", 
        "Caprock Evidence": "Unknown", 
        "Structural Risk Evidence": "Unknown", 
        "Screening Verdict": "Unknown"
    }
    
    for line in answer.split('\n'):
        line_clean = line.strip()
        if line_clean.startswith("Evidence Type:"): parsed_fields["Evidence Type"] = line_clean.replace("Evidence Type:", "").strip()
        elif line_clean.startswith("Caprock Evidence:"): parsed_fields["Caprock Evidence"] = line_clean.replace("Caprock Evidence:", "").strip()
        elif line_clean.startswith("Structural Risk Evidence:"): parsed_fields["Structural Risk Evidence"] = line_clean.replace("Structural Risk Evidence:", "").strip()
        elif line_clean.startswith("Screening Verdict:"): parsed_fields["Screening Verdict"] = line_clean.replace("Screening Verdict:", "").strip()

    final_reports.append({
        "interval_id": interval_id,
        "ml_predictions": target,
        "evidence_confidence": evidence_conf,
        "rag_geological_evaluation": answer,
        "retrieved_evidence": retrieved_sources
    })

    summary_table_data.append({
        "Interval ID": interval_id,
        "Tier": tier.strip(),
        "Thickness": f"{thickness} {unit}",
        "ML Confidence": round(ml_conf, 4),
        "Petrophysics": petro_str,
        "Evidence Confidence": evidence_conf,
        "Evidence Type": parsed_fields["Evidence Type"],
        "Caprock Evidence": parsed_fields["Caprock Evidence"],
        "Risk Evidence": parsed_fields["Structural Risk Evidence"],
        "Screening Verdict": parsed_fields["Screening Verdict"]
    })
    
    time.sleep(2)

FINAL_JSON_PATH = os.path.join(REPORT_DIR, "thesis_gradient_ccs_reports.json")
with open(FINAL_JSON_PATH, 'w') as f:
    json.dump({"gradient_candidates_evaluation": final_reports}, f, indent=4)

summary_df = pd.DataFrame(summary_table_data)
FINAL_EXCEL_PATH = os.path.join(REPORT_DIR, "thesis_gradient_summary_table.xlsx")
summary_df.to_excel(FINAL_EXCEL_PATH, index=False)

print("\n" + "="*50)
print(f" GRADIENT RAG PIPELINE COMPLETE!")
print(f" JSON Saved to: {FINAL_JSON_PATH}")
print(f" Summary Excel saved to: {FINAL_EXCEL_PATH}")
print("="*50)

  STEP 3: DIVERSE GRADIENT RAG SCREENING & THESIS EXPORT


 Enter your DeepSeek API Key:  ········


 Initializing DeepSeek LLM...
 Loaded 20 intervals. Evaluating targets via RAG Pipeline...

--------------------------------------------------
  DEEP EVALUATING: Tarim_Target_6 |  BEST
    (ML Conf: 0.8728, Thick: 63.0, Petro: avg_GR: 65.558, avg_RHOB: 2.468, avg_NPHI: 0.154, avg_PEF: 3.911, avg_DTC: 69.208)
 Evaluating via DeepSeek...

 Report:
Evidence Type: Robust
Caprock Evidence: Strong
Structural Risk Evidence: Moderate Risk
Screening Verdict: Medium-High
Reason: The machine learning prediction confidence of 0.8728 is strong, and the petrophysical log suite is internally consistent with a viable reservoir: a moderate GR (65.558 API) suggests a mixed lithology, a robust density (avg_RHOB: 2.468 g/cc) and neutron porosity (avg_NPHI: 0.154 v/v) indicate competent rock with appreciable pore space, and a low DTC (69.208 µs/ft) supports rigidity. The primary volumetric constraint is the thickness proxy of 63.0 samples, which converts to approximately 9.6 meters, marginally below the 10